# Sistema de Recomendacao de Livros (Otimizado)

## IAA012 - Frameworks de IA
### Especializacao em Inteligencia Artificial Aplicada - UFPR/SEPT

---

Este notebook implementa um **Sistema de Recomendacao de Livros** utilizando **Filtragem Colaborativa** com **Redes Neurais** e **Embeddings**.

**Otimizacoes implementadas:**
- Batch size otimizado para GPU
- Regularizacao L2 nos embeddings
- Callbacks: EarlyStopping + ReduceLROnPlateau
- Estruturas de dados eficientes (dicionarios para lookup O(1))
- Predicao vetorizada com batch otimizado

## 1. Importacao das Bibliotecas

In [ ]:
# Compatibilidade TensorFlow/Keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Verificar GPU
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponivel: {tf.config.list_physical_devices('GPU')}")

## 2. Carregamento e Pre-processamento dos Dados

In [ ]:
# Carregar dados
df = pd.read_csv('Base_livros.csv')

print(f"Dataset: {df.shape[0]:,} registros")
print(f"Usuarios unicos: {df['ID_usuario'].nunique():,}")
print(f"Livros unicos: {df['ISBN'].nunique():,}")
df.head()

In [ ]:
# Codificacao eficiente usando pd.Categorical (mais rapido que LabelEncoder)
df['user_id'] = pd.Categorical(df['ID_usuario']).codes
df['book_id'] = pd.Categorical(df['ISBN']).codes

n_users = df['user_id'].nunique()
n_books = df['book_id'].nunique()

print(f"Usuarios (encoded): {n_users:,}")
print(f"Livros (encoded): {n_books:,}")

In [ ]:
# Criar mapeamentos eficientes (lookup O(1))
book_info = df.drop_duplicates('book_id').set_index('book_id')[['ISBN', 'Titulo', 'Autor']].to_dict('index')
user_to_code = dict(zip(df['ID_usuario'], df['user_id']))

# Livros avaliados por usuario (para filtragem rapida)
user_rated_books = df.groupby('user_id')['book_id'].apply(set).to_dict()

print("Mapeamentos criados com sucesso!")

In [ ]:
# Preparar dados
user_input = df['user_id'].values
book_input = df['book_id'].values
ratings = df['Notas'].values.astype(np.float32)

# Normalizacao (centralizacao na media - melhor para regressao)
ratings_mean = ratings.mean()
ratings_std = ratings.std()
ratings_norm = (ratings - ratings_mean) / ratings_std

print(f"Media das notas: {ratings_mean:.2f}")
print(f"Desvio padrao: {ratings_std:.2f}")

In [ ]:
# Divisao treino/teste (stratified shuffle)
indices = np.arange(len(ratings))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

u_train, u_test = user_input[train_idx], user_input[test_idx]
b_train, b_test = book_input[train_idx], book_input[test_idx]
r_train, r_test = ratings_norm[train_idx], ratings_norm[test_idx]

print(f"Treino: {len(r_train):,} | Teste: {len(r_test):,}")

## 3. Construcao do Modelo Otimizado

In [ ]:
# Hiperparametros
EMBEDDING_DIM = 50
L2_REG = 1e-5
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 1024  # Otimizado para GPU
EPOCHS = 30

In [ ]:
def build_model(n_users, n_books, emb_dim=50, l2_reg=1e-5, dropout=0.2):
    """
    Modelo NCF otimizado com regularizacao L2 e Dropout.
    """
    # Inputs
    user_inp = Input(shape=(1,), name='user_input')
    book_inp = Input(shape=(1,), name='book_input')
    
    # Embeddings com regularizacao L2
    user_emb = Embedding(
        n_users, emb_dim, 
        embeddings_regularizer=l2(l2_reg),
        name='user_embedding'
    )(user_inp)
    user_vec = Flatten()(user_emb)
    
    book_emb = Embedding(
        n_books, emb_dim,
        embeddings_regularizer=l2(l2_reg),
        name='book_embedding'
    )(book_inp)
    book_vec = Flatten()(book_emb)
    
    # Concatenacao + Camadas densas
    x = Concatenate()([user_vec, book_vec])
    x = Dense(256, activation='relu')(x)
    x = Dropout(dropout)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(dropout)(x)
    x = Dense(64, activation='relu')(x)
    
    # Saida linear (regressao)
    output = Dense(1, activation='linear', name='output')(x)
    
    return Model(inputs=[user_inp, book_inp], outputs=output)

# Criar modelo
model = build_model(n_users, n_books, EMBEDDING_DIM, L2_REG, DROPOUT_RATE)

# Compilar
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

model.summary()

## 4. Treinamento com Callbacks

In [ ]:
# Callbacks otimizados
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print(f"Treinando: {EPOCHS} epochs, batch_size={BATCH_SIZE}")

In [ ]:
# Treinamento
history = model.fit(
    [u_train, b_train], r_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=([u_test, b_test], r_test),
    callbacks=callbacks,
    verbose=1
)

## 5. Graficos de Avaliacao do Modelo (Loss)

### Interpretacao dos Graficos de Loss

Os graficos de loss sao fundamentais para entender o comportamento do modelo durante o treinamento:

1. **Convergencia:** A diminuicao da loss indica que o modelo esta aprendendo
2. **Overfitting:** Se val_loss aumenta enquanto loss diminui, ha overfitting
3. **Underfitting:** Se ambas as curvas estabilizam em valores altos, o modelo e muito simples
4. **Bom ajuste:** Curvas proximas que diminuem juntas indicam boa generalizacao

In [ ]:
# Graficos de Loss e MAE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], 'b-', label='Treino', linewidth=2)
axes[0].plot(history.history['val_loss'], 'r-', label='Validacao', linewidth=2)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Funcao de Perda (Loss)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], 'b-', label='Treino', linewidth=2)
axes[1].plot(history.history['val_mae'], 'r-', label='Validacao', linewidth=2)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('MAE')
axes[1].set_title('Erro Medio Absoluto (MAE)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Metricas finais
print("\n" + "="*50)
print("METRICAS FINAIS")
print("="*50)
print(f"Loss (treino):     {history.history['loss'][-1]:.4f}")
print(f"Loss (validacao):  {history.history['val_loss'][-1]:.4f}")
print(f"MAE (treino):      {history.history['mae'][-1]:.4f}")
print(f"MAE (validacao):   {history.history['val_mae'][-1]:.4f}")

# MAE na escala original
mae_original = history.history['val_mae'][-1] * ratings_std
print(f"\nMAE na escala original (0-10): {mae_original:.2f}")

### Explicacao dos Resultados de Loss

**O que observamos:**

- **Queda inicial rapida:** O modelo aprende rapidamente os padroes mais obvios
- **Estabilizacao:** Apos algumas epocas, a loss estabiliza indicando convergencia
- **Gap treino/validacao:** Um pequeno gap e normal; gap grande indica overfitting

**Regularizacao utilizada:**
- L2 nos embeddings: previne que os vetores cresam demais
- Dropout: desliga neuronios aleatoriamente, forcando generalizacao
- ReduceLROnPlateau: reduz learning rate quando a validacao estagna

## 6. Sistema de Recomendacao

In [ ]:
def recomendar_livros(usuario_id, top_n=10):
    """
    Gera recomendacoes de livros para um usuario.
    
    Parametros:
    - usuario_id: ID original do usuario (ex: 276725)
    - top_n: numero de recomendacoes
    
    Retorna:
    - DataFrame com livros recomendados e notas previstas
    """
    # Verificar se usuario existe
    if usuario_id not in user_to_code:
        print(f"Usuario {usuario_id} nao encontrado!")
        return None
    
    user_code = user_to_code[usuario_id]
    
    # Livros ja avaliados pelo usuario
    livros_avaliados = user_rated_books.get(user_code, set())
    
    # Livros para predizer (nao avaliados)
    all_books = np.arange(n_books)
    livros_novos = np.array([b for b in all_books if b not in livros_avaliados])
    
    if len(livros_novos) == 0:
        print(f"Usuario {usuario_id} ja avaliou todos os livros!")
        return None
    
    # Predicao vetorizada (eficiente)
    users_array = np.full(len(livros_novos), user_code)
    preds = model.predict([users_array, livros_novos], batch_size=4096, verbose=0).flatten()
    
    # Desnormalizar
    preds_original = preds * ratings_std + ratings_mean
    
    # Top-N
    top_idx = np.argsort(preds_original)[::-1][:top_n]
    top_books = livros_novos[top_idx]
    top_notas = preds_original[top_idx]
    
    # Criar DataFrame com resultados
    resultados = []
    for book_code, nota in zip(top_books, top_notas):
        info = book_info.get(book_code, {})
        resultados.append({
            'ISBN': info.get('ISBN', 'N/A'),
            'Titulo': info.get('Titulo', 'N/A'),
            'Autor': info.get('Autor', 'N/A'),
            'Nota_Prevista': round(nota, 2)
        })
    
    return pd.DataFrame(resultados)

In [ ]:
def mostrar_historico(usuario_id, top_n=5):
    """Mostra os livros avaliados pelo usuario."""
    if usuario_id not in user_to_code:
        print(f"Usuario {usuario_id} nao encontrado!")
        return None
    
    historico = df[df['ID_usuario'] == usuario_id][['Titulo', 'Autor', 'Notas']]
    return historico.sort_values('Notas', ascending=False).head(top_n)

## 7. Exemplo de Recomendacao para um Usuario

### Demonstracao do Sistema de Recomendacao

Vamos selecionar um usuario e gerar recomendacoes personalizadas.

In [ ]:
# Encontrar usuarios mais ativos
usuarios_ativos = df.groupby('ID_usuario').size().sort_values(ascending=False)
print("Top 5 usuarios mais ativos:")
print(usuarios_ativos.head())

# Selecionar usuario para demonstracao
USUARIO_EXEMPLO = usuarios_ativos.index[0]
print(f"\nUsuario selecionado: {USUARIO_EXEMPLO}")
print(f"Total de avaliacoes: {usuarios_ativos[USUARIO_EXEMPLO]}")

In [ ]:
# Historico do usuario
print(f"\n{'='*70}")
print(f"HISTORICO DO USUARIO {USUARIO_EXEMPLO}")
print(f"{'='*70}")
print("\nLivros com melhores notas:")
display(mostrar_historico(USUARIO_EXEMPLO, top_n=5))

In [ ]:
# Gerar recomendacoes
print(f"\n{'='*70}")
print(f"RECOMENDACOES PARA O USUARIO {USUARIO_EXEMPLO}")
print(f"{'='*70}")

recomendacoes = recomendar_livros(USUARIO_EXEMPLO, top_n=10)
print("\nTop 10 livros recomendados:")
display(recomendacoes)

In [ ]:
# Visualizacao das recomendacoes
if recomendacoes is not None:
    plt.figure(figsize=(12, 6))
    
    titulos = [t[:35] + '...' if len(str(t)) > 35 else t for t in recomendacoes['Titulo']]
    notas = recomendacoes['Nota_Prevista'].values
    
    bars = plt.barh(range(len(titulos)), notas, color='steelblue', edgecolor='navy')
    plt.yticks(range(len(titulos)), titulos)
    plt.xlabel('Nota Prevista')
    plt.title(f'Top 10 Recomendacoes para Usuario {USUARIO_EXEMPLO}')
    plt.xlim(0, 10)
    plt.gca().invert_yaxis()
    
    for bar, nota in zip(bars, notas):
        plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{nota:.1f}', va='center')
    
    plt.tight_layout()
    plt.show()

### Interpretacao das Recomendacoes

**Como o modelo gera recomendacoes:**

1. **Embeddings de Usuario:** Capturam as preferencias latentes do usuario
2. **Embeddings de Livro:** Capturam caracteristicas latentes dos livros
3. **Rede Neural:** Aprende interacoes complexas entre usuario e livro
4. **Predicao:** A nota prevista indica a probabilidade de o usuario gostar do livro

**Por que funciona:**
- Usuarios com gostos similares terao embeddings proximos
- Livros com caracteristicas similares terao embeddings proximos
- O modelo aprende a "combinar" usuarios e livros compativeis

In [ ]:
# Testar com outro usuario
print("\n" + "="*70)
print("RECOMENDACOES PARA OUTRO USUARIO")
print("="*70)

outro_usuario = usuarios_ativos.index[5]
print(f"\nUsuario: {outro_usuario} ({usuarios_ativos[outro_usuario]} avaliacoes)")

print("\nHistorico:")
display(mostrar_historico(outro_usuario, top_n=3))

print("\nRecomendacoes:")
display(recomendar_livros(outro_usuario, top_n=5))

## 8. Avaliacao Final

In [ ]:
# Avaliacao no conjunto de teste
test_loss, test_mae = model.evaluate([u_test, b_test], r_test, verbose=0)
test_mae_original = test_mae * ratings_std

print("="*50)
print("AVALIACAO FINAL DO MODELO")
print("="*50)
print(f"Loss (MSE) no teste: {test_loss:.4f}")
print(f"MAE no teste: {test_mae:.4f}")
print(f"MAE na escala 0-10: {test_mae_original:.2f}")
print(f"\nInterpretacao: O modelo erra, em media, {test_mae_original:.2f} pontos")

In [ ]:
# Grafico: Predicoes vs Valores Reais
preds_test = model.predict([u_test, b_test], batch_size=4096, verbose=0).flatten()
preds_original = preds_test * ratings_std + ratings_mean
reais_original = r_test * ratings_std + ratings_mean

plt.figure(figsize=(8, 6))
plt.scatter(reais_original, preds_original, alpha=0.2, s=5)
plt.plot([0, 10], [0, 10], 'r--', linewidth=2, label='Predicao Perfeita')
plt.xlabel('Nota Real')
plt.ylabel('Nota Prevista')
plt.title('Predicoes vs Valores Reais')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 10.5)
plt.ylim(-0.5, 10.5)
plt.tight_layout()
plt.show()

## 9. Conclusao

### Resumo das Otimizacoes

| Otimizacao | Beneficio |
|------------|-----------||
| `pd.Categorical` | Encoding 3-5x mais rapido que LabelEncoder |
| Batch size 1024 | Melhor utilizacao de GPU |
| L2 Regularization | Previne overfitting nos embeddings |
| Dropout | Melhora generalizacao |
| ReduceLROnPlateau | Ajuste automatico do learning rate |
| Dicionarios para lookup | O(1) vs O(n) na busca de informacoes |
| Predicao vetorizada | Batch processing eficiente |

### Resultados

O modelo consegue prever notas de livros com erro medio de aproximadamente 2-3 pontos na escala de 0-10, o que e um resultado razoavel para sistemas de recomendacao baseados apenas em avaliacoes.

In [ ]:
print("\nSistema de Recomendacao de Livros - Concluido!")
print(f"\nResumo:")
print(f"- Usuarios: {n_users:,}")
print(f"- Livros: {n_books:,}")
print(f"- Embedding dim: {EMBEDDING_DIM}")
print(f"- Epochs treinados: {len(history.history['loss'])}")
print(f"- MAE final: {test_mae_original:.2f}")